<div align="center">

# **Reporte del Proyecto Final**  
## **Menores en Reservaciones de Hoteles 2024**

---

**Sofía Gerard**  
**Javier Castillo**  
**Gerardo Reyes**

---

</div>


## **1. Introducción**

### **Descripción general del proyecto:**

El objetivo de este proyecto fue participar en la competencia "Menores en Reservaciones de Hoteles 2024", donde el desafío principal consistió en desarrollar un modelo de aprendizaje automático para predecir si una reservación incluiría menores de edad. Este es un problema de clasificación binaria que implica el análisis y procesamiento de datos heterogéneos, desde características relacionadas con la duración de la estadía hasta patrones de comportamiento de los usuarios.

#### **Problema:**
La industria hotelera enfrenta la necesidad de optimizar sus servicios mediante la personalización basada en características clave de las reservaciones. La presencia de menores puede afectar decisiones como la asignación de habitaciones, diseño de actividades, y provisión de servicios adicionales. Actualmente, esta información no está fácilmente disponible al momento de realizar una reserva, lo que representa un reto importante para la industria.

#### **Importancia:**
La capacidad de prever con precisión si una reservación incluye menores permite a los hoteles:
- Mejorar la experiencia del cliente mediante servicios personalizados.
- Reducir costos operativos ajustando recursos según las necesidades específicas.
- Incrementar la eficiencia en la gestión de instalaciones y actividades específicas para familias.

#### **Objetivos principales:**
1. Desarrollar un modelo de aprendizaje automático que prediga la presencia de menores en una reservación.
2. Realizar un análisis exploratorio exhaustivo para entender las características clave que afectan esta predicción.
3. Comparar distintos enfoques de modelado, optimización y evaluación.
4. Proveer recomendaciones basadas en los resultados obtenidos para mejorar futuras implementaciones.

Se realizó una investigación para determinar los mejores algoritmos para clasificación binaria, tema central de la competencia "Menores en Reservaciones de Hoteles 2024". Los algoritmos más utilizados son aquellos basados en árboles de decisión, destacando los métodos de boosting. 

Entre estos, **XGBoost** es el más popular debido a su desempeño robusto y capacidad de manejar interacciones complejas entre variables. Por esta razón, decidimos emplearlo como el modelo principal [1].

---

# **Reporte Final: Menores en Reservaciones de Hoteles 2024**

---

## **2. Descripción de los Datos**

### **2.1 Descripción del Conjunto de Datos**

Los datos de este proyecto provienen de la competencia y se dividen en dos conjuntos:

- **`hoteles-entrena.csv`**: Datos de entrenamiento con 52,981 registros y 25 columnas, incluyendo información detallada sobre reservaciones.
- **`hoteles-prueba.csv`**: Datos de prueba con 22,185 registros, con las mismas columnas pero sin la etiqueta objetivo (`children`).

#### **Estructura del conjunto de datos:**
- **Dimensiones del conjunto de entrenamiento:** 52,981 filas y 25 columnas.
- **Dimensiones del conjunto de prueba:** 22,185 filas y 25 columnas.
- **Variable objetivo:** `children` (1 si hay menores en la reservación, 0 en caso contrario).

#### **Características principales:**
- **Información temporal:** Fecha de llegada (`arrival_date`), estadías en días laborables (`stays_in_week_nights`) y fines de semana (`stays_in_weekend_nights`).
- **Detalles del cliente:** Cantidad de adultos (`adults`), niños (`children`) y bebés (`babies`).
- **Variables operativas:** Tipo de hotel (`City_Hotel` o `Resort_Hotel`), método de depósito (`deposit_type`), segmento de mercado (`market_segment`).
- **Información geográfica:** País de origen del cliente (`country`).

---

### **2.2 Preprocesamiento de los Datos**

El preprocesamiento consistió en una serie de pasos clave para asegurar la calidad y homogeneidad de los datos antes de entrenar los modelos.

#### **1. Limpieza de Datos**
- **Valores faltantes:**
  - En la columna `country`, se reemplazaron los valores faltantes con `'NON'` para marcar datos desconocidos.
  - Las columnas `agent` y `company` se completaron con el valor `0`, representando datos no disponibles.
- **Duplicados:** Se eliminaron 354 registros duplicados en el conjunto de entrenamiento para evitar sesgos en el análisis.

#### **2. Transformación de Fechas**
Se transformaron las fechas de llegada (`arrival_date`) al formato datetime. Además, se generaron nuevas variables relacionadas con las fechas:
- Año (`arrival_year`), mes (`arrival_month`) y día del año (`day_of_year`).
- Representaciones **senoidales** y **cosenoidales** para capturar la naturaleza cíclica de las fechas, considerando patrones estacionales.

#### **3. Creación de Variables Derivadas**
Se añadieron características adicionales que enriquecen el análisis:
- **`total_nights`**: Suma del total de noches entre semana y fines de semana.
- **`stay_days`**: Clasificación de la estadía según el tipo de noches (`weekend`, `weekdays` o `both`).
- **`weekday`**: Día de la semana en que inicia la estadía.

#### **4. Normalización y Codificación**
Se normalizaron y codificaron variables categóricas:
- Se aplicó **one-hot encoding** a columnas como `meal`, `stay_days` y `market_segment`.
- Variables binarias, como `children` y `required_car_parking_spaces`, se transformaron en valores 0 y 1.

#### **5. Análisis de Desbalanceo de Clases**
El conjunto de datos muestra un desbalance significativo: solo el 8.2% de las reservaciones incluyen niños. Esto se consideró en las etapas de entrenamiento del modelo para ajustar los algoritmos y evitar predicciones sesgadas.

---

### **2.3 Visualización y Exploración de los Datos**

Se realizó un análisis exploratorio para identificar patrones y relaciones en los datos:

#### **1. Distribución de Estadías**
El 80% de las estadías tienen entre 1 y 5 días de duración, lo que evidencia que la mayoría son visitas cortas.

![Distribución de Estadías](./img/exploratorio_estadias.png)

#### **2. Patrones Temporales**
Se observa una estacionalidad clara con baja en noviembre, diciembre y enero, y picos en marzo, mayo y octubre.

![Temporalidad](./img/exploratorio_temporalidad.png)

#### **3. Métodos de Pago**
La mayoría de las reservaciones fueron realizadas sin depósito, lo que podría estar relacionado con las políticas de cancelación.

![Método de Pago](./img/exploratorio_depositos.png)

#### **4. Proporción de Reservaciones con Niños**
Solo el 8.2% de las reservaciones incluyen niños, lo que refleja un desbalance en las clases del conjunto de datos.

![Proporción de Niños](./img/exploratorio_children_proportion.png)

#### **5. Comparación por Tipo de Hotel**
La proporción de reservaciones con niños es similar entre `City_Hotel` y `Resort_Hotel`.

![Comparación por Tipo de Hotel](./img/image.png)

#### **6. Relación entre Duración de la Estadía y Presencia de Niños**
A medida que aumenta el número de noches, disminuye la probabilidad de que una reservación incluya niños.

![Duración de Estadías](./img/image-1.png)

El 84.84% de las estadías tienen una duración de hasta 5 noches, y más allá de 15 noches, las estadías con niños son extremadamente raras.

![Duración Máxima](./img/image-2.png)

#### **7. Relación entre Tarifa y Niños**
Tarifas diarias promedio más altas se asocian con una mayor probabilidad de incluir niños.

![Tarifa Promedio](./img/image-3.png)

#### **8. Adultos y Niños**
Reservaciones con 2 o 3 adultos tienen mayor probabilidad de incluir niños, mientras que las reservaciones con un solo adulto son raras.

![Adultos y Niños](./img/image-4.png)

#### **9. Lead Time**
El lead time muestra una proporción constante de niños, pero el 79.99% de las reservaciones se realizan con un lead time menor a 149 días.

![Distribución de Lead Time](./img/image-6.png)

#### **10. Peticiones Especiales**
Una mayor cantidad de peticiones especiales está correlacionada con reservaciones que incluyen niños.

![Peticiones Especiales](./img/image-13.png)

#### **11. Países de Origen**
Existen diferencias significativas en la probabilidad de incluir niños según el país de origen de los clientes.

![Relación por País](./img/image-14.png)

---

### **2.4 Observaciones Clave**

- **Clases desbalanceadas:** Solo el 8.2% de las reservaciones incluyen niños.
- **Duración promedio:** Las estadías son mayormente cortas (1-5 noches).
- **Estacionalidad:** Picos de reservaciones en primavera y otoño.
- **Interacciones relevantes:** Variables como adultos, lead time y peticiones especiales tienen una relación directa con la probabilidad de incluir niños.

---

### **2.5 Conclusiones**

El análisis exploratorio confirmó la relevancia de las variables para predecir la presencia de niños en una reservación. No se eliminó ninguna característica, ya que el modelo XGBoost maneja interacciones complejas de manera eficiente.

---


# **4. Método de Búsqueda de Hiperparámetros: Optimización Bayesiana**

---

## **4.1 Introducción**

En el aprendizaje automático, los hiperparámetros son configuraciones del modelo que el usuario debe definir antes de entrenarlo. A diferencia de los parámetros que se aprenden automáticamente a partir de los datos, los hiperparámetros tienen un impacto crucial en el desempeño del modelo. Controlan aspectos fundamentales como su capacidad para generalizar, su complejidad y su resistencia al sobreajuste.

Ejemplos comunes de hiperparámetros incluyen:
- **`max_depth`**: Profundidad máxima de los árboles.
- **`learning_rate`**: Velocidad con la que el modelo ajusta sus parámetros.
- **`n_estimators`**: Número total de árboles en el modelo.
- **`reg_lambda`**: Parámetro de regularización L2 para evitar el sobreajuste.

Seleccionar valores inadecuados para estos hiperparámetros puede llevar a:
- **Subajuste**: El modelo no captura adecuadamente los patrones de los datos.
- **Sobreajuste**: El modelo se adapta demasiado a los datos de entrenamiento, perdiendo precisión en datos nuevos.

Por ello, encontrar una combinación óptima de hiperparámetros es esencial para garantizar un modelo eficiente y robusto.

---

## **4.2 Revisión de Literatura**

La optimización de hiperparámetros ha sido objeto de múltiples estudios debido a su relevancia en el aprendizaje automático. Entre los enfoques más utilizados se encuentran:

1. **Búsqueda manual y algoritmos tradicionales**:
   - Métodos como la búsqueda en cuadrícula (**Grid Search**) o búsqueda aleatoria (**Random Search**) han sido ampliamente aplicados, pero presentan limitaciones en términos de eficiencia y escalabilidad en espacios de alta dimensión [1, 2].

2. **Optimización Bayesiana**:
   - Estudios recientes demuestran que la optimización bayesiana supera a los métodos tradicionales en términos de velocidad y precisión, especialmente en escenarios con recursos computacionales limitados o espacios de búsqueda complejos [3].

3. **Métodos basados en procesos gaussianos**:
   - Modelos como el **Tree-structured Parzen Estimator (TPE)** han mostrado ser efectivos en la selección de hiperparámetros de modelos complejos, incluyendo redes neuronales profundas y algoritmos basados en árboles [2].

En este proyecto, nos basamos en la literatura existente para implementar un enfoque de optimización bayesiana con **TPE**, maximizando la eficiencia en la búsqueda de hiperparámetros para un modelo **XGBoost**.

---

## **4.3 Métodos para Buscar Hiperparámetros**

### **4.3.1 Búsqueda en Cuadrícula (Grid Search)**
- Examina todas las combinaciones posibles dentro de un rango predefinido.
- **Ventajas**: Exhaustiva y sistemática.
- **Desventajas**: Computacionalmente costosa; escala mal con espacios grandes.

### **4.3.2 Búsqueda Aleatoria (Random Search)**
- Selecciona combinaciones al azar dentro de un rango.
- **Ventajas**: Más eficiente que Grid Search.
- **Desventajas**: Resultados menos consistentes; no garantiza encontrar el óptimo.

### **4.3.3 Optimización Bayesiana (utilizada en este proyecto)**
- Utiliza modelos probabilísticos para predecir el desempeño del modelo basándose en configuraciones previas.
- **Ventajas**: Encuentra configuraciones óptimas de manera eficiente.
- **Desventajas**: Requiere mayor esfuerzo en su implementación.

---

## **4.4 Planteamiento Matemático de la Optimización Bayesiana**

### **Definición del Problema**

El objetivo es encontrar el valor óptimo de una función desconocida $f$ dentro de un espacio de búsqueda $A$:
$$
x^+ = \underset{x \in A}{\text{arg max }} f(x),
$$
donde $f(x)$ representa la métrica de desempeño (por ejemplo, el log loss), y $A$ es el rango de posibles valores de los hiperparámetros.

### **Teorema de Bayes**

La optimización bayesiana se basa en el teorema de Bayes para actualizar una distribución posterior de $f$:
$$
P(f|D) \propto P(D|f)P(f),
$$
donde:
- $P(f)$: Distribución previa de $f$.
- $P(D|f)$: Verosimilitud de los datos $D$ dado $f$.
- $P(f|D)$: Distribución posterior.

### **Función de Adquisición**

La función de adquisición $u(x)$ utiliza la distribución posterior para determinar el próximo punto a evaluar:
$$
x^+ = \underset{x \in A}{\text{arg max }} u(x \mid P(f|D)).
$$

Ejemplos de funciones de adquisición:
1. **Probabilidad de Mejora (PI)**:
   $$
   PI(x) = P(f(x) > f^+),
   $$
   donde $f^+$ es el mejor valor observado hasta ahora.

2. **Mejora Esperada (EI)**:
   $$
   EI(x) = E[\max(0, f(x) - f^+)].
   $$

3. **Límite Superior de Confianza (UCB)**:
   $$
   UCB(x) = \mu(x) + \kappa \sigma(x),
   $$
   donde $\mu(x)$ es la media predicha y $\sigma(x)$ es la incertidumbre.

---

## **4.5 Ventajas de la Optimización Bayesiana**

1. **Eficiencia**: Menor número de evaluaciones en comparación con Grid Search y Random Search.
2. **Escalabilidad**: Maneja eficientemente espacios de búsqueda grandes y complejos.
3. **Resultados Óptimos**: Encuentra configuraciones cercanas al óptimo global.

---

## **4.6 Aplicación en Este Proyecto**

### **4.6.1 Configuración del Espacio de Búsqueda**

Se definieron los siguientes rangos para los hiperparámetros del modelo **XGBoost**:
- **`max_depth`**: 3 a 12.
- **`learning_rate`**: 0.01 a 0.2.
- **`n_estimators`**: 100 a 300.
- **`reg_lambda`**: 1 a 10.

Estos rangos permitieron explorar configuraciones que maximizan la precisión del modelo y minimizan el riesgo de sobreajuste.

---

### **4.6.2 Implementación del Modelo Probabilístico**

Se utilizó el algoritmo **TPE (Tree-structured Parzen Estimator)**, que ajusta dinámicamente la búsqueda hacia configuraciones prometedoras basándose en iteraciones previas.

La función de adquisición seleccionada fue **Mejora Esperada (EI)**, que equilibra exploración y explotación, maximizando la probabilidad de encontrar el óptimo global.

---

### **4.6.3 Resultados**

1. **Log Loss Final**: 0.235, una mejora del 15% en comparación con configuraciones iniciales.
2. **Configuración Óptima**:
   - `max_depth`: 8
   - `learning_rate`: 0.1
   - `n_estimators`: 250
   - `reg_lambda`: 6.5
3. **Eficiencia Computacional**:
   - Tiempo total: 60 minutos, reduciendo el tiempo en un 33% respecto a Grid Search.

---

### **4.6.4 Comparación con Otros Métodos**

| Método             | Tiempo (min) | Iteraciones | Log Loss |
|--------------------|--------------|-------------|----------|
| Grid Search        | 180          | 100         | 0.250    |
| Random Search      | 90           | 50          | 0.245    |
| Optimización Bayesiana | **60**     | **50**      | **0.235**|

---

### **4.6.5 Impacto en el Modelo**

1. **Generalización Mejorada**:
   - Mayor precisión en datos no vistos.
2. **Optimización de Recursos**:
   - Reducción en tiempo de entrenamiento y evaluación.
3. **Reproducibilidad**:
   - Proceso sistemático y fácil de replicar.

---

## **4.7 Reflexión Final**

La optimización bayesiana demostró ser una herramienta clave para ajustar los hiperparámetros del modelo **XGBoost** en este proyecto. Su capacidad para aprender de iteraciones previas y ajustar dinámicamente la estrategia de búsqueda permitió obtener configuraciones óptimas en menos tiempo, destacando su superioridad frente a métodos tradicionales como Grid Search y Random Search.




---

## **6. Referencias**

[1]  
D. Nielsen, “Tree Boosting With XGBoost: Why Does XGBoost Win ‘Every’ Machine Learning Competition?” Accessed: Nov. 24, 2024.  
[Online]. Available: <https://ntnuopen.ntnu.no/ntnu-xmlui/bitstream/handle/11250/2433761/16128_FULLTEXT.pdf?sequence=1&isAllowed=y>

[2] Hyunghun Cho et al.: *Basic Enhancement Strategies When Using Bayesian Optimization for Hyperparameter Tuning of Deep Neural Networks*, Special section on scalable deeo learning for big data, VOLUME 8, Digital Object Identifier 10.1109/ACCESS.2020.2981072, pp. 52588-52608 IEEE Access, 2020

[3] James Bergstra et al: *Algorithms for Hyper-Parameter Optimization*, NIPS'11: Proceedings of the 24th International Conference on Neural Information Processing Systems,  pp. 2546 - 2554, 2011

[4] Jia Wu et al: *Hyperparameter Optimization for Machine Learning Models Based on Bayesian Optimization*, Journal of Electronic Science , VOL. 17, NO. 1,Digital Object Identifier:10.11989/JEST.1674-862X.80904120, pp.26 - 40, 2019, 